In [29]:
import numpy as np

Q = np.loadtxt('QUBO_Matrix_1.txt')
n = Q.shape[0]

with open('matrix_1.qubo', 'w') as f:
    f.write(f"{n}\n")  # n=8 vars at top (matches test.txt style)
    # Triplets 0-indexed i <= j (sprs-safe)
    for i in range(n):
        for j in range(i, n):
            f.write(f"{i} {j} {Q[i,j]/10000:.10f}\n")

print("converted matrix is saved")


converted matrix is saved


In [30]:
# Branch and Bound

In [1]:
import hercules

problem = hercules.read_qubo("matrix_1.qubo")

x_soln, obj, *_ = hercules.solve_branch_bound(
    problem,
    timeout=10.0,
    sub_problem_solver="clarabel_lp"
)

print("Exact solution:", x_soln)
print("Exact energy:", obj)


Hercules: A Rust-based Branch and Bound Solver for QUBO
Version number 0.2.0
Problem size: 8
Fixed variables: 1
------------------------------------------------------
Nodes Visited | Best Solution | Lower Bound | Gap (%)
1             | 0.00000000    | -0.75245627 | 7524562.66913784
2             | -0.57895000   | -0.70191379 | 21.23947109  
3             | -0.57895000   | -0.66589658 | 15.01823608  
4             | -0.57895000   | -0.63761289 | 10.13281011  
5             | -0.57895000   | -0.61536835 | 6.29052260   
6             | -0.57895000   | -0.59697670 | 3.11374250   
7             | -0.57895000   | -0.58418920 | 0.90496455   
7             | -0.57895000   | -0.57895000 | 0.00000000   
------------------------------------------------------
Branch and Bound Solver Finished
Best Solution: [1, 1, 0, 1, 1, 1, 1, 1]
Best Solution Value: -0.5789500000000001
Nodes Solved: 7
Nodes Processed: 13
Nodes Visited: 15
Time to Solve: 0.009824514389038086
-------------------------------------

# Another Approach via the mixed local search heuristic with

In [32]:
import hercules
import random
import time

# Read QUBO (assumes matrix_1.qubo format works)
problem = hercules.read_qubo('matrix_1.qubo')
num_x = problem[-1]  # Extract num vars from problem object

print(f"Problem size: {num_x}")

# Random initial point
random.seed(time.time())
x_0 = [random.randint(0, 1) for _ in range(num_x)]

# Timing
start = time.time()

# Mixed Local Search: 100 iterations (tunable)
x_soln, obj = hercules.mls(problem, x_0, 100)

end = time.time()

# Results
print('Solution vector:', x_soln)
print('Energy:', obj, 'in', end - start, 'seconds')
print('Binary string:', ''.join(map(str, x_soln)))


Problem size: 8
Solution vector: [0, 0, 1, 0, 1, 1, 1, 0]
Energy: -0.4891 in 0.0006558895111083984 seconds
Binary string: 00101110
